In [1]:
import os
import time

from pyspark.sql import SparkSession

os.environ['OBJC_DISABLE_INITIALIZE_FORK_SAFETY'] = 'YES'

try:
    existing_spark = SparkSession.getActiveSession()
    if existing_spark:
        existing_spark.stop()
except:
    pass

for key in list(os.environ.keys()):
    if 'SPARK' in key or 'JAVA_OPTS' in key:
        del os.environ[key]

# --- 2. Cluster Configuration ---
# Format: local-cluster[num_workers, cores_per_worker, memory_per_worker_in_MB]
NUM_EXECUTORS = 2
CORES_PER_EXECUTOR = 6
MEMORY_PER_EXECUTOR_MB = 4096

MASTER_URL = f"local-cluster[{NUM_EXECUTORS}, {CORES_PER_EXECUTOR}, {MEMORY_PER_EXECUTOR_MB}]"

print(f"Running in mode: {MASTER_URL}")

sp_s = (SparkSession.builder
    .master(MASTER_URL)
    .appName("LocalClusterTest")
    .config("spark.driver.memory", "4g")
    .config("spark.executor.memory", "4g")
    .config("spark.executor.cores", "6")
    .config("spark.executor.instances", NUM_EXECUTORS)
    .config("spark.memory.fraction", "0.6")
    .config("spark.sql.shuffle.partitions", "4")  # For tests, less than the default 200
    .getOrCreate()
)

sp_s.sparkContext.setLogLevel("WARN")

# --- 3. Configuration Check ---
print("Session created.")
print(f"Driver Memory Config: {sp_s.conf.get('spark.driver.memory')}")
print(f"Executor Memory Config: {sp_s.conf.get('spark.executor.memory')}")

# Check the number of executors (may take a couple of seconds to start)
time.sleep(3)
num_executors = len(sp_s.sparkContext.parallelize(range(10), NUM_EXECUTORS).glom().collect())
print(f"📊 Active executors (checked via RDD): {num_executors}")

# --- 4. Distribution Test (Example) ---
# To make sure the task went to executors, not stayed on the driver
def print_executor_info(iterator):
    import os
    # Get the executor ID from the process environment variables
    executor_id = os.environ.get('SPARK_EXECUTOR_ID', 'Driver/Local')
    process_id = os.getpid()
    return [f"Executor ID: {executor_id}, PID: {process_id}"]

# Create a dataframe and apply a transformation
df = sp_s.range(0, 10, 1, 4)  # 4 partitions
result = df.rdd.mapPartitions(print_executor_info).collect()

print("\n🖥️ Where tasks were executed:")
for line in result:
    print(line)

sp_s

Running in mode: local-cluster[2, 6, 4096]


26/08/28 16:10:18 WARN Utils: Your hostname, MacBook-Pro-Danil.local resolves to a loopback address: 127.0.0.1; using 10.246.43.29 instead (on interface en0)
26/08/28 16:10:18 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/08/28 16:10:18 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Session created.
Driver Memory Config: 4g
Executor Memory Config: 4g


📊 Active executors (checked via RDD): 2

🖥️ Where tasks were executed:
Executor ID: Driver/Local, PID: 5537
Executor ID: Driver/Local, PID: 5536
Executor ID: Driver/Local, PID: 5546
Executor ID: Driver/Local, PID: 5545


# AA test tutorial 
AA test is important part of randomized controlled experiment, for example AB test. 

The objectives of the AA test are to verify the assumption of uniformity of samples as a result of the applied partitioning method, to select the best partition from the available ones, and to verify the applicability of statistical criteria for checking uniformity. 

For example, there is a hypothesis about the absence of dependence of features on each other. If this hypothesis is not followed, the AA test will fail.

[Wiki AA test](https://github.com/sb-ai-lab/HypEx/wiki/%D0%90%D0%90-Test) with more detailed description of terms for AA test.

<ul>
  <li><a href="#creation-of-a-new-test-dataset-with-synthetic-data">Creation of a new test dataset with synthetic data.
  <li><a href="#one-split-of-aa-test">One split of AA test.
  <li><a href="#aa-test">AA test.
  <li><a href="#aa-test-with-stratification">AA test with stratification.
</ul>

In [2]:
from hypex import AATest
from hypex.dataset import (
    ConstGroupRole,
    Dataset,
    InfoRole,
    StratificationRole,
    TargetRole,
    TreatmentRole,
)
from hypex.utils import BackendsEnum, create_test_data


/Users/danilsamsutdinov/HypEx/.venv/lib/python3.11/site-packages/pyspark/pandas/__init__.py:50: UserWarning: 'PYARROW_IGNORE_TIMEZONE' environment variable was not set. It is required to set this environment variable to '1' in both driver and executor sides if you use pyarrow>=2.0.0. pandas-on-Spark will set it for you but it does not work if there is a Spark context already launched.
  warnings.warn(


## Creation of a new test dataset with synthetic data. 

In order to be able to work with our data in HypEx, first we need to convert it into `dataset`. It is important to mark the data fields by assigning the appropriate `roles`:
- TargetRole: a role for columns that contain features or predictor variables. Our split will be based on them. Applied by default if the role is not specified for the column.
- TreatmentRole: a role for columns that show the treatment or intervention.
- InfoRole: a role for columns that contain information about the data, such as user IDs. 

In [3]:
data = Dataset(
    roles={
        "user_id": InfoRole(int),
        "pre_spends": TargetRole(),
        "post_spends": TargetRole(),
        "gender": StratificationRole(str),
    },
    data=create_test_data(),
    session=sp_s,
    backend=BackendsEnum.spark
)
data

,user_id,signup_month,treat,pre_spends,post_spends,age,gender,industry
0,0.0,6.0,1.0,482.0,484.444444,44.0,M,E-commerce
1,1.0,0.0,0.0,481.5,425.111111,45.0,M,Logistics
2,2.0,0.0,0.0,494.5,417.222222,65.0,F,Logistics
3,3.0,0.0,0.0,486.5,419.777778,44.0,M,E-commerce
4,4.0,0.0,0.0,491.0,421.888889,24.0,M,E-commerce
...,...,...,...,...,...,...,...,...
9995,9995.0,3.0,1.0,483.0,517.333333,39.0,M,Logistics
9996,9996.0,4.0,1.0,488.5,511.444444,65.0,F,Logistics
9997,9997.0,10.0,1.0,459.5,435.888889,48.0,F,E-commerce
9998,9998.0,7.0,1.0,488.0,479.333333,43.0,F,Logistics


In [4]:
data.roles

{'user_id': Info(<class 'int'>),
 'pre_spends': Target(<class 'float'>),
 'post_spends': Target(<class 'float'>),
 'gender': Stratification(<class 'str'>),
 'signup_month': Default(<class 'float'>),
 'treat': Default(<class 'float'>),
 'age': Default(<class 'float'>),
 'industry': Default(<class 'str'>)}

## AA test
Then we run the experiment on our prepared dataset, wrapped into ExperimentData. In this case we select one of the pre-assembled pipeline, AA_TEST.
We can set the number of iterations for simple execution. In this case the random states are the numbers of each iteration.

In [5]:
test = AATest(n_iterations=10)
result = test.execute(data)

100%|██████████| 10/10 [00:22<00:00,  2.21s/it]
[DEBUG _compute_weighted_pvalues] pval_cols = ['pre_spends TTest p-value test_1', 'post_spends TTest p-value test_1', 'pre_spends KSTest p-value test_1', 'post_spends KSTest p-value test_1', 'mean TTest p-value all', 'mean KSTest p-value all']
[DEBUG _compute_weighted_pvalues] col='pre_spends TTest p-value test_1', lookup_key='pre_spends TTest test_1', weight=0.95, NaN=0/10, values=[0.11769972339513825, 0.052410190248373144, 0.6732329472683363, 0.9306272718097068, 0.16173360145871307, 0.4860729100486433, 0.6389652372454337, 0.6996533977761284, 0.35822152999072054, 0.08355131015633357]
[DEBUG _compute_weighted_pvalues] col='post_spends TTest p-value test_1', lookup_key='post_spends TTest test_1', weight=0.95, NaN=0/10, values=[0.23734422654941964, 0.17956614243590335, 0.43055071096293507, 0.10395508694133135, 0.8711561145921224, 0.16902593374948274, 0.3851907860183159, 0.8882209744333717, 0.7398744443604306, 0.4426561591673792]
[DEBUG _com

In [6]:
result.resume

,feature,group,TTest aa test,KSTest aa test,TTest best split,KSTest best split,result,control mean,test mean,difference,difference %
0,post_spends,test_1,OK,OK,OK,OK,OK,452.064844,451.413107,-0.651736,-0.144169
1,pre_spends,test_1,OK,OK,OK,OK,OK,486.895287,487.061070,0.165784,0.034049


**Interpretation of AA test results**

Each row in the table corresponds to a target feature being tested for equality between the control and test groups. Two statistical tests are used:

- **TTest**: tests if means are statistically different.
- **KSTest**: tests if distributions differ.

The `OK` / `NOT OK` labels show whether the difference is statistically significant. A `NOT OK` result indicates a possible imbalance.

Typical threshold:
- If p-value < 0.05 → `NOT OK` (statistically significant difference)
- If p-value ≥ 0.05 → `OK` (no significant difference)

If any metric has a `NOT OK` status in the `AA test` column, it means at least one iteration showed significant difference.


In [7]:
result.aa_score

,score,pass
pre_spends TTest test_1,0.95,True
post_spends TTest test_1,0.95,True
pre_spends KSTest test_1,0.95,True
post_spends KSTest test_1,0.95,True


**Interpreting `aa_score`**

This output shows p-values and the overall pass/fail status for each test type and feature. A high p-value (close to 1.0) means the test passed — the groups are similar.

- `score`: p-value of the statistical test.
- `pass`: True if no iterations showed significant differences.

Note: Even if the average p-value is high, the `pass` might still be False if at least one of the iterations had a p-value < 0.05.


In [8]:
result.best_split

,user_id,signup_month,treat,pre_spends,post_spends,age,gender,industry,split
2,2,0.0,0.0,494.5,417.222222,65.0,F,Logistics,control
4,4,0.0,0.0,491.0,421.888889,24.0,M,E-commerce,control
5,5,2.0,1.0,474.5,532.666667,35.0,M,Logistics,test_1
8,8,0.0,0.0,496.5,420.333333,64.0,M,Logistics,control
12,12,8.0,1.0,476.5,464.777778,52.0,M,Logistics,test_1
...,...,...,...,...,...,...,...,...,...
8996,9146,3.0,1.0,478.0,527.777778,30.0,F,Logistics,control
8997,9156,0.0,0.0,474.5,417.222222,64.0,M,Logistics,test_1
8998,9158,2.0,1.0,491.0,514.666667,47.0,F,E-commerce,test_1
8999,9159,7.0,1.0,494.5,460.666667,59.0,M,E-commerce,control


**About `best_split`**

This shows the best found split of the dataset, where control and test groups are as similar as possible in terms of target metrics.

You can use this split for future modeling or as a validation check before proceeding to actual experiments.


In [9]:
result.best_split_statistic

,feature,group,control mean,test mean,difference,difference %,TTest pass,TTest p-value,KSTest pass,KSTest p-value
0,post_spends,test_1,452.064844,451.413107,-0.651736,-0.144169,OK,0.430551,OK,0.619872
1,pre_spends,test_1,486.895287,487.061070,0.165784,0.034049,OK,0.673233,OK,0.978495


**Understanding `best_split_statistic`**

This table contains detailed statistics for the best (most balanced) split found across all iterations. You can compare:

- Mean values in control vs test group.
- Absolute and relative differences.
- p-values for both tests.

Ideally, all rows should have `OK` in both TTest and KSTest columns, and small difference values (<1%).

In [10]:
result.experiments

,splitter_id,pre_spends GroupDifference control mean test_1,pre_spends GroupDifference test mean test_1,pre_spends GroupDifference difference test_1,pre_spends GroupDifference difference % test_1,post_spends GroupDifference control mean test_1,post_spends GroupDifference test mean test_1,post_spends GroupDifference difference test_1,post_spends GroupDifference difference % test_1,pre_spends TTest p-value test_1,...,post_spends TTest pass test_1,pre_spends KSTest p-value test_1,pre_spends KSTest pass test_1,post_spends KSTest p-value test_1,post_spends KSTest pass test_1,mean TTest p-value,mean TTest pass,mean KSTest p-value,mean KSTest pass,mean test score
0,AASplitter┴rs 0┴,487.281750,486.666704,-0.615046,-0.126220,451.256611,452.233679,0.977068,0.216522,0.117700,...,False,0.627058,False,0.461328,False,0.177522,0.0,0.544193,0.0,0.421969
1,AASplitter┴rs 1┴,486.602497,487.365051,0.762554,0.156710,451.191999,452.301741,1.109742,0.245958,0.052410,...,False,0.771481,False,0.178564,False,0.115988,0.0,0.475023,0.0,0.355344
2,AASplitter┴rs 2┴,486.895287,487.061070,0.165784,0.034049,452.064844,451.413107,-0.651736,-0.144169,0.673233,...,False,0.978495,False,0.619872,False,0.551892,0.0,0.799184,0.0,0.716753
3,AASplitter┴rs 3┴,486.995334,486.961111,-0.034223,-0.007027,452.410872,451.066568,-1.344304,-0.297142,0.930627,...,False,0.781739,False,0.050677,False,0.517291,0.0,0.416208,0.0,0.449902
4,AASplitter┴rs 4┴,487.255488,486.705422,-0.550066,-0.112891,451.671197,451.805305,0.134108,0.029691,0.161734,...,False,0.704531,False,0.563671,False,0.516445,0.0,0.634101,0.0,0.594882
5,AASplitter┴rs 5┴,486.844779,487.118700,0.273921,0.056265,451.184648,452.322134,1.137486,0.252111,0.486073,...,False,0.500148,False,0.123770,False,0.327549,0.0,0.311959,0.0,0.317156
6,AASplitter┴rs 6┴,487.069512,486.885073,-0.184439,-0.037867,452.094173,451.376157,-0.718017,-0.158820,0.638965,...,False,0.981417,False,0.124130,False,0.512078,0.0,0.552773,0.0,0.539208
7,AASplitter┴rs 7┴,486.902184,487.053844,0.151661,0.031148,451.797064,451.680848,-0.116216,-0.025723,0.699653,...,False,0.130245,False,0.808165,False,0.793937,0.0,0.469205,0.0,0.577449
8,AASplitter┴rs 8┴,487.158391,486.797216,-0.361175,-0.074139,451.875736,451.601213,-0.274523,-0.060752,0.358222,...,False,0.102798,False,0.973542,False,0.549048,0.0,0.538170,0.0,0.541796
9,AASplitter┴rs 9┴,486.639362,487.319581,0.680218,0.139779,451.422576,452.057340,0.634763,0.140614,0.083551,...,False,0.484883,False,0.803770,False,0.263104,0.0,0.644326,0.0,0.517252


# AA Test with random states

We can also adjust some of the preset parameters of the experiment by assigning them to the respective params of the experiment. I.e. here we set the range of the random states we want to run our AA test for. 

In [11]:
data = Dataset(
    roles={
        "user_id": InfoRole(int),
        "pre_spends": TargetRole(),
        "post_spends": TargetRole(),
        "gender": StratificationRole(str),
    }, 
    data=create_test_data(),
    session=sp_s,
    backend=BackendsEnum.spark
)
data

,user_id,signup_month,treat,pre_spends,post_spends,age,gender,industry
0,0.0,8.0,1.0,488.0,463.111111,55.0,M,Logistics
1,1.0,0.0,0.0,452.5,410.222222,38.0,F,Logistics
2,2.0,0.0,0.0,472.5,416.666667,21.0,F,E-commerce
3,3.0,0.0,0.0,475.5,424.222222,40.0,M,E-commerce
4,4.0,3.0,1.0,479.5,513.888889,33.0,M,E-commerce
...,...,...,...,...,...,...,...,...
9995,9995.0,3.0,1.0,487.0,513.111111,27.0,F,E-commerce
9996,9996.0,0.0,0.0,506.5,414.777778,49.0,F,E-commerce
9997,9997.0,0.0,0.0,475.0,423.111111,21.0,M,Logistics
9998,9998.0,0.0,0.0,470.5,415.0,65.0,M,E-commerce


In [12]:
test = AATest(random_states=[56, 72, 2, 43])
result = test.execute(data)

100%|██████████| 4/4 [00:07<00:00,  1.86s/it]
[DEBUG _compute_weighted_pvalues] pval_cols = ['pre_spends TTest p-value test_1', 'post_spends TTest p-value test_1', 'pre_spends KSTest p-value test_1', 'post_spends KSTest p-value test_1', 'mean TTest p-value all', 'mean KSTest p-value all']
[DEBUG _compute_weighted_pvalues] col='pre_spends TTest p-value test_1', lookup_key='pre_spends TTest test_1', weight=0.95, NaN=0/4, values=[0.7370681758088948, 0.22348493943494352, 0.6418670518800951, 0.5281424806435913]
[DEBUG _compute_weighted_pvalues] col='post_spends TTest p-value test_1', lookup_key='post_spends TTest test_1', weight=0.95, NaN=0/4, values=[0.8007289917280114, 0.6487543139348622, 0.6076541683036144, 0.569951431124361]
[DEBUG _compute_weighted_pvalues] col='pre_spends KSTest p-value test_1', lookup_key='pre_spends KSTest test_1', weight=0.95, NaN=0/4, values=[0.9478465248938973, 0.4126906297815784, 0.9708295341791803, 0.513304420030315]
[DEBUG _compute_weighted_pvalues] col='post_

In [13]:
result.resume

,feature,group,TTest aa test,KSTest aa test,TTest best split,KSTest best split,result,control mean,test mean,difference,difference %
0,post_spends,test_1,OK,OK,OK,OK,OK,451.619892,451.829230,0.209338,0.046353
1,pre_spends,test_1,OK,OK,OK,OK,OK,487.034336,487.168215,0.133879,0.027489


In [14]:
result.aa_score

,score,pass
pre_spends TTest test_1,0.95,True
post_spends TTest test_1,0.95,True
pre_spends KSTest test_1,0.95,True
post_spends KSTest test_1,0.95,True


In [15]:
result.best_split

,user_id,signup_month,treat,pre_spends,post_spends,age,gender,industry,split
2,2,0.0,0.0,472.5,416.666667,21.0,F,E-commerce,control
4,4,3.0,1.0,479.5,513.888889,33.0,M,E-commerce,test_1
5,5,0.0,0.0,458.0,425.333333,30.0,M,E-commerce,control
8,8,0.0,0.0,477.0,428.333333,44.0,F,Logistics,control
12,12,3.0,1.0,514.0,508.222222,56.0,F,Logistics,test_1
...,...,...,...,...,...,...,...,...,...
8996,9994,6.0,1.0,470.0,480.222222,59.0,F,Logistics,test_1
8997,9995,3.0,1.0,487.0,513.111111,27.0,F,E-commerce,control
8998,9997,0.0,0.0,475.0,423.111111,21.0,M,Logistics,test_1
8999,9998,0.0,0.0,470.5,415.0,65.0,M,E-commerce,control


In [16]:
result.best_split_statistic

,feature,group,control mean,test mean,difference,difference %,TTest pass,TTest p-value,KSTest pass,KSTest p-value
0,post_spends,test_1,451.619892,451.829230,0.209338,0.046353,OK,0.800729,OK,0.821491
1,pre_spends,test_1,487.034336,487.168215,0.133879,0.027489,OK,0.737068,OK,0.947847


In [17]:
result.experiments

,splitter_id,pre_spends GroupDifference control mean test_1,pre_spends GroupDifference test mean test_1,pre_spends GroupDifference difference test_1,pre_spends GroupDifference difference % test_1,post_spends GroupDifference control mean test_1,post_spends GroupDifference test mean test_1,post_spends GroupDifference difference test_1,post_spends GroupDifference difference % test_1,pre_spends TTest p-value test_1,...,post_spends TTest pass test_1,pre_spends KSTest p-value test_1,pre_spends KSTest pass test_1,post_spends KSTest p-value test_1,post_spends KSTest pass test_1,mean TTest p-value,mean TTest pass,mean KSTest p-value,mean KSTest pass,mean test score
0,AASplitter┴rs 56┴,487.034336,487.168215,0.133879,0.027489,451.619892,451.829230,0.209338,0.046353,0.737068,...,False,0.947847,False,0.821491,False,0.768899,0.0,0.884669,0.0,0.846079
1,AASplitter┴rs 72┴,486.856282,487.341593,0.485311,0.099683,451.912222,451.534513,-0.377709,-0.083580,0.223485,...,False,0.412691,False,0.557752,False,0.436120,0.0,0.485221,0.0,0.468854
2,AASplitter┴rs 2┴,487.192649,487.007225,-0.185424,-0.038060,451.935302,451.509560,-0.425743,-0.094204,0.641867,...,False,0.970830,False,0.724789,False,0.624761,0.0,0.847809,0.0,0.773460
3,AASplitter┴rs 43┴,486.973942,487.225449,0.251507,0.051647,451.958649,451.487549,-0.471100,-0.104235,0.528142,...,False,0.513304,False,0.783614,False,0.549047,0.0,0.648459,0.0,0.615322


# AA Test with stratification

Depending on your requirements it is possible to stratify the data. You can set `stratification=True` and `StratificationRole` in `Dataset` to run it with stratification.

Stratified AA tests ensure that both groups (control/test) have the same proportions of categories (e.g. same % of genders or regions). This prevents imbalances in categorical features that can distort results.

Make sure to assign `StratificationRole` to relevant columns in your dataset before enabling stratification.

In [18]:
data = Dataset(
    roles={
        "user_id": InfoRole(int),
        "pre_spends": TargetRole(),
        "post_spends": TargetRole(),
        "gender": StratificationRole(str),
    }, 
    data=create_test_data(),
    session=sp_s,
    backend=BackendsEnum.spark
)
data

,user_id,signup_month,treat,pre_spends,post_spends,age,gender,industry
0,0.0,5.0,1.0,477.5,494.777778,26.0,F,E-commerce
1,1.0,0.0,0.0,497.5,425.555556,33.0,M,Logistics
2,2.0,5.0,1.0,486.5,502.222222,66.0,M,E-commerce
3,3.0,0.0,0.0,495.0,415.888889,57.0,F,E-commerce
4,4.0,0.0,0.0,482.0,418.0,18.0,M,Logistics
...,...,...,...,...,...,...,...,...
9995,9995.0,11.0,1.0,484.0,425.0,34.0,M,Logistics
9996,9996.0,0.0,0.0,481.0,421.0,27.0,M,E-commerce
9997,9997.0,8.0,1.0,501.0,460.333333,65.0,M,E-commerce
9998,9998.0,0.0,0.0,473.5,410.333333,54.0,F,Logistics


In [19]:
test = AATest(random_states=[56, 72, 2, 43], stratification=True)
result = test.execute(data)

100%|██████████| 4/4 [00:08<00:00,  2.12s/it]
[DEBUG _compute_weighted_pvalues] pval_cols = ['pre_spends TTest p-value test_1', 'post_spends TTest p-value test_1', 'pre_spends KSTest p-value test_1', 'post_spends KSTest p-value test_1', 'mean TTest p-value all', 'mean KSTest p-value all']
[DEBUG _compute_weighted_pvalues] col='pre_spends TTest p-value test_1', lookup_key='pre_spends TTest test_1', weight=0.95, NaN=0/4, values=[0.8130083668313348, 0.744127095953639, 0.7813239825038201, 0.46857413122055636]
[DEBUG _compute_weighted_pvalues] col='post_spends TTest p-value test_1', lookup_key='post_spends TTest test_1', weight=0.95, NaN=0/4, values=[0.8234077679422324, 0.1621791542746909, 0.4212566000252812, 0.40949834201395907]
[DEBUG _compute_weighted_pvalues] col='pre_spends KSTest p-value test_1', lookup_key='pre_spends KSTest test_1', weight=0.95, NaN=0/4, values=[0.5657441503548161, 0.7572905554668996, 0.7108453919636115, 0.4167616112329016]
[DEBUG _compute_weighted_pvalues] col='pos

In [20]:
result.resume

,feature,group,TTest aa test,KSTest aa test,TTest best split,KSTest best split,result,control mean,test mean,difference,difference %
0,post_spends,test_1,OK,OK,OK,OK,OK,451.497181,451.681394,0.184214,0.040801
1,pre_spends,test_1,OK,OK,OK,OK,OK,487.374604,487.281121,-0.093483,-0.019181


In [21]:
result.aa_score

,score,pass
pre_spends TTest test_1,0.95,True
post_spends TTest test_1,0.95,True
pre_spends KSTest test_1,0.95,True
post_spends KSTest test_1,0.95,True


In [22]:
result.best_split

,user_id,signup_month,treat,pre_spends,post_spends,age,gender,industry,split
0,0,5.0,1.0,477.5,494.777778,26.0,F,E-commerce,test_1
1,1,0.0,0.0,497.5,425.555556,33.0,M,Logistics,test_1
2,2,5.0,1.0,486.5,502.222222,66.0,M,E-commerce,control
3,3,0.0,0.0,495.0,415.888889,57.0,F,E-commerce,test_1
4,4,0.0,0.0,482.0,418.0,18.0,M,Logistics,test_1
...,...,...,...,...,...,...,...,...,...
8996,9995,11.0,1.0,484.0,425.0,34.0,M,Logistics,control
8997,9996,0.0,0.0,481.0,421.0,27.0,M,E-commerce,test_1
8998,9997,8.0,1.0,501.0,460.333333,65.0,M,E-commerce,test_1
8999,9998,0.0,0.0,473.5,410.333333,54.0,F,Logistics,control


In [23]:
result.best_split_statistic

,feature,group,control mean,test mean,difference,difference %,TTest pass,TTest p-value,KSTest pass,KSTest p-value
0,post_spends,test_1,451.497181,451.681394,0.184214,0.040801,OK,0.823408,OK,0.887783
1,pre_spends,test_1,487.374604,487.281121,-0.093483,-0.019181,OK,0.813008,OK,0.565744


In [24]:
result.experiments

,splitter_id,pre_spends GroupDifference control mean test_1,pre_spends GroupDifference test mean test_1,pre_spends GroupDifference difference test_1,pre_spends GroupDifference difference % test_1,post_spends GroupDifference control mean test_1,post_spends GroupDifference test mean test_1,post_spends GroupDifference difference test_1,post_spends GroupDifference difference % test_1,pre_spends TTest p-value test_1,...,post_spends TTest pass test_1,pre_spends KSTest p-value test_1,pre_spends KSTest pass test_1,post_spends KSTest p-value test_1,post_spends KSTest pass test_1,mean TTest p-value,mean TTest pass,mean KSTest p-value,mean KSTest pass,mean test score
0,AASplitterWithStratification┴rs 56┴,487.374604,487.281121,-0.093483,-0.019181,451.497181,451.681394,0.184214,0.040801,0.813008,...,False,0.565744,False,0.887783,False,0.818208,0.0,0.726764,0.0,0.757245
1,AASplitterWithStratification┴rs 72┴,487.262774,487.391741,0.128967,0.026468,451.016909,452.170461,1.153553,0.255767,0.744127,...,False,0.757291,False,0.271521,False,0.453153,0.0,0.514406,0.0,0.493988
2,AASplitterWithStratification┴rs 2┴,487.272152,487.381836,0.109684,0.022510,451.922743,451.259004,-0.663739,-0.146870,0.781324,...,False,0.710845,False,0.730488,False,0.601290,0.0,0.720667,0.0,0.680875
3,AASplitterWithStratification┴rs 43┴,487.184106,487.470490,0.286384,0.058784,451.251509,451.932195,0.680686,0.150844,0.468574,...,False,0.416762,False,0.734055,False,0.439036,0.0,0.575408,0.0,0.529951


# AA Test by samples 

Depending on your requirements and size of data it is possible to estimate AA test on samples the data. You can set `sample_size=size` to run it. 

In [25]:
data = Dataset(
    roles={
        "user_id": InfoRole(int),
        "pre_spends": TargetRole(),
        "post_spends": TargetRole(),
        "gender": StratificationRole(str),
    },
    data=create_test_data(),
    session=sp_s,
    backend=BackendsEnum.spark
)
data

,user_id,signup_month,treat,pre_spends,post_spends,age,gender,industry
0,0.0,0.0,0.0,469.5,411.0,36.0,M,Logistics
1,1.0,0.0,0.0,503.0,423.0,32.0,F,E-commerce
2,2.0,7.0,1.0,497.5,474.333333,30.0,M,Logistics
3,3.0,0.0,0.0,508.0,439.0,20.0,M,E-commerce
4,4.0,0.0,0.0,478.5,419.555556,18.0,F,E-commerce
...,...,...,...,...,...,...,...,...
9995,9995.0,7.0,1.0,476.5,487.444444,33.0,F,E-commerce
9996,9996.0,10.0,1.0,511.0,437.444444,27.0,F,Logistics
9997,9997.0,0.0,0.0,512.5,440.222222,62.0,F,Logistics
9998,9998.0,0.0,0.0,496.0,413.222222,35.0,F,Logistics


In [26]:
test = AATest(n_iterations=10, sample_size=0.3)
result = test.execute(data)

100%|██████████| 10/10 [00:17<00:00,  1.74s/it]
[DEBUG _compute_weighted_pvalues] pval_cols = ['pre_spends TTest p-value control', 'post_spends TTest p-value control', 'pre_spends KSTest p-value control', 'post_spends KSTest p-value control', 'mean TTest p-value all', 'mean KSTest p-value all']
[DEBUG _compute_weighted_pvalues] col='pre_spends TTest p-value control', lookup_key='pre_spends TTest control', weight=0.95, NaN=0/10, values=[0.18356513448396897, 0.9503547529205676, 0.7918402220437113, 0.6160777523371428, 0.06891425638661489, 0.21444523379659364, 0.737187324730084, 0.05229187309150994, 0.038055589027683724, 0.5441554832026443]
[DEBUG _compute_weighted_pvalues] col='post_spends TTest p-value control', lookup_key='post_spends TTest control', weight=0.95, NaN=0/10, values=[0.09871723038047005, 0.4127410824001182, 0.4118652923789228, 0.29787374621028595, 0.5159128815270649, 0.8225484052658811, 0.7223995154839715, 0.674286262888658, 0.681948259724416, 0.7603648867601827]
[DEBUG _c

In [27]:
result.resume

,feature,group,TTest aa test,KSTest aa test,TTest best split,KSTest best split,result,control mean,test mean,difference,difference %
0,post_spends,control,OK,OK,OK,OK,OK,451.917360,452.239005,0.321645,0.071173
1,pre_spends,control,OK,OK,OK,OK,OK,487.348603,487.495092,0.146488,0.030058


In [28]:
result.aa_score

,score,pass
pre_spends TTest control,0.95,True
post_spends TTest control,0.95,True
pre_spends KSTest control,0.95,True
post_spends KSTest control,0.95,True


In [29]:
result.best_split

,user_id,signup_month,treat,pre_spends,post_spends,age,gender,industry,split
2,2,7.0,1.0,497.5,474.333333,30.0,M,Logistics,control
4,4,0.0,0.0,478.5,419.555556,18.0,F,E-commerce,control
5,5,3.0,1.0,479.5,514.777778,47.0,M,Logistics,test_1
8,8,0.0,0.0,464.0,414.0,50.0,F,E-commerce,control
12,12,4.0,1.0,460.5,513.333333,55.0,F,Logistics,test_1
...,...,...,...,...,...,...,...,...,...
8996,9994,10.0,1.0,482.5,457.444444,25.0,M,Logistics,test_1
8997,9995,7.0,1.0,476.5,487.444444,33.0,F,E-commerce,control
8998,9997,0.0,0.0,512.5,440.222222,62.0,F,Logistics,test_1
8999,9998,0.0,0.0,496.0,413.222222,35.0,F,Logistics,control


In [30]:
result.best_split_statistic

,feature,group,control mean,test mean,difference,difference %,TTest pass,TTest p-value,KSTest pass,KSTest p-value
0,post_spends,control,451.917360,452.239005,0.321645,0.071173,OK,0.722400,OK,0.558631
1,pre_spends,control,487.348603,487.495092,0.146488,0.030058,OK,0.737187,OK,0.960845


In [31]:
result.experiments

,splitter_id,pre_spends GroupDifference control mean control,pre_spends GroupDifference test mean control,pre_spends GroupDifference difference control,pre_spends GroupDifference difference % control,post_spends GroupDifference control mean control,post_spends GroupDifference test mean control,post_spends GroupDifference difference control,post_spends GroupDifference difference % control,pre_spends TTest p-value control,...,post_spends TTest pass control,pre_spends KSTest p-value control,pre_spends KSTest pass control,post_spends KSTest p-value control,post_spends KSTest pass control,mean TTest p-value,mean TTest pass,mean KSTest p-value,mean KSTest pass,mean test score
0,AASplitter┴rs 0┴,487.050146,487.627177,0.577031,0.118475,451.109814,452.595711,1.485897,0.329387,0.183565,...,False,0.392731,False,0.164274,False,0.141141,0.0,0.278503,0.0,0.232715
1,AASplitter┴rs 1┴,487.469986,487.443078,-0.026908,-0.005520,452.650331,451.916135,-0.734196,-0.162199,0.950355,...,False,0.755001,False,0.636909,False,0.681548,0.0,0.695955,0.0,0.691153
2,AASplitter┴rs 2┴,487.531365,487.416653,-0.114711,-0.023529,451.627333,452.367100,0.739767,0.163800,0.791840,...,False,0.478455,False,0.674221,False,0.601853,0.0,0.576338,0.0,0.584843
3,AASplitter┴rs 3┴,487.606255,487.386639,-0.219615,-0.045039,452.809889,451.864223,-0.945666,-0.208844,0.616078,...,False,0.846283,False,0.348915,False,0.456976,0.0,0.597599,0.0,0.550724
4,AASplitter┴rs 4┴,486.889057,487.686034,0.796977,0.163688,451.726457,452.316888,0.590431,0.130705,0.068914,...,False,0.138254,False,0.473275,False,0.292414,0.0,0.305764,0.0,0.301314
5,AASplitter┴rs 5┴,487.839662,487.293087,-0.546575,-0.112040,451.997571,452.202377,0.204807,0.045311,0.214445,...,False,0.114361,False,0.905945,False,0.518497,0.0,0.510153,0.0,0.512934
6,AASplitter┴rs 6┴,487.348603,487.495092,0.146488,0.030058,451.917360,452.239005,0.321645,0.071173,0.737187,...,False,0.960845,False,0.558631,False,0.729793,0.0,0.759738,0.0,0.749757
7,AASplitter┴rs 7┴,486.858098,487.704549,0.846451,0.173860,451.876548,452.256776,0.380228,0.084144,0.052292,...,False,0.066571,False,0.813045,False,0.363289,0.0,0.439808,0.0,0.414302
8,AASplitter┴rs 8┴,488.083827,487.180108,-0.903719,-0.185156,451.883872,452.254238,0.370366,0.081960,0.038056,...,False,0.187208,False,0.793404,False,0.360002,0.5,0.490306,0.0,0.446871
9,AASplitter┴rs 9┴,487.267851,487.531340,0.263489,0.054075,451.951625,452.226440,0.274814,0.060806,0.544155,...,False,0.167593,False,0.973476,False,0.652260,0.0,0.570534,0.0,0.597776


# AATest with Target Role for a categorical feature

It is possible to assign Target Role to categorical features. A categorical feature can also be the target or outcome variable. In this case, the Chi-square test is added to the pipeline of AATest.

In [32]:
data = Dataset(
    roles={
        "user_id": InfoRole(int),
        "treat": TreatmentRole(int),
        "pre_spends": TargetRole(),
        "post_spends": TargetRole(),
        "gender": TargetRole(str)
    }, data=create_test_data(),
)
data

,user_id,signup_month,treat,pre_spends,post_spends,age,gender,industry
0,0.0,0.0,0.0,496.0,410.555556,57.0,F,Logistics
1,1.0,11.0,1.0,481.0,438.333333,43.0,M,E-commerce
2,2.0,1.0,1.0,540.5,517.777778,27.0,F,E-commerce
3,3.0,11.0,1.0,481.5,439.0,57.0,F,E-commerce
4,4.0,10.0,1.0,519.5,441.222222,34.0,F,Logistics
...,...,...,...,...,...,...,...,...
9995,9995.0,0.0,0.0,508.0,422.111111,23.0,F,E-commerce
9996,9996.0,0.0,0.0,505.5,407.111111,24.0,F,Logistics
9997,9997.0,0.0,0.0,482.5,400.666667,59.0,M,E-commerce
9998,9998.0,0.0,0.0,485.0,429.555556,66.0,F,Logistics


In [33]:
test = AATest(n_iterations=10)
result = test.execute(data)

100%|██████████| 10/10 [00:02<00:00,  4.02it/s]
[DEBUG _compute_weighted_pvalues] pval_cols = ['pre_spends TTest p-value test_1', 'post_spends TTest p-value test_1', 'pre_spends KSTest p-value test_1', 'post_spends KSTest p-value test_1', 'mean TTest p-value all', 'mean KSTest p-value all']
[DEBUG _compute_weighted_pvalues] col='pre_spends TTest p-value test_1', lookup_key='pre_spends TTest test_1', weight=0.95, NaN=0/10, values=[0.04515028664201466, 0.6648856079570142, 0.4793075869571978, 0.5165781813983803, 0.695790065443102, 0.6106745157473207, 0.5812827936309723, 0.16594574162317377, 0.7506006339559198, 0.2646827143256581]
[DEBUG _compute_weighted_pvalues] col='post_spends TTest p-value test_1', lookup_key='post_spends TTest test_1', weight=0.95, NaN=0/10, values=[0.6259411237631212, 0.28796612978867026, 0.6923468457573576, 0.04221278204616523, 0.5462753777808553, 0.7359280672062123, 0.9000990230924069, 0.33233149052380817, 0.8537018181167462, 0.9697014519903355]
[DEBUG _compute_we

In [34]:
result.resume

,feature,group,TTest aa test,KSTest aa test,TTest best split,KSTest best split,result,control mean,test mean,difference,difference %
0,post_spends,test_1,OK,OK,OK,OK,OK,452.354799,452.250845,-0.103954,-0.022981
1,pre_spends,test_1,OK,OK,OK,OK,OK,487.307029,487.523820,0.216791,0.044487


In [35]:
result.aa_score

,score,pass
pre_spends TTest test_1,0.95,True
post_spends TTest test_1,0.95,True
pre_spends KSTest test_1,0.95,True
post_spends KSTest test_1,0.95,True


In [36]:
result.best_split

,user_id,signup_month,treat,pre_spends,post_spends,age,gender,industry,split
0,0.0,0.0,0.0,496.0,410.555556,57.0,F,Logistics,control
1,1.0,11.0,1.0,481.0,438.333333,43.0,M,E-commerce,test_1
2,2.0,1.0,1.0,540.5,517.777778,27.0,F,E-commerce,test_1
3,3.0,11.0,1.0,481.5,439.0,57.0,F,E-commerce,control
4,4.0,10.0,1.0,519.5,441.222222,34.0,F,Logistics,test_1
...,...,...,...,...,...,...,...,...,...
9995,9995.0,0.0,0.0,508.0,422.111111,23.0,F,E-commerce,control
9996,9996.0,0.0,0.0,505.5,407.111111,24.0,F,Logistics,control
9997,9997.0,0.0,0.0,482.5,400.666667,59.0,M,E-commerce,test_1
9998,9998.0,0.0,0.0,485.0,429.555556,66.0,F,Logistics,test_1


In [37]:
result.best_split_statistic

,feature,group,control mean,test mean,difference,difference %,TTest pass,TTest p-value,KSTest pass,KSTest p-value
0,post_spends,test_1,452.354799,452.250845,-0.103954,-0.022981,OK,0.900099,OK,0.976042
1,pre_spends,test_1,487.307029,487.523820,0.216791,0.044487,OK,0.581283,OK,0.831604


In [38]:
result.experiments

,splitter_id,pre_spends GroupDifference control mean test_1,pre_spends GroupDifference test mean test_1,pre_spends GroupDifference difference test_1,pre_spends GroupDifference difference % test_1,post_spends GroupDifference control mean test_1,post_spends GroupDifference test mean test_1,post_spends GroupDifference difference test_1,post_spends GroupDifference difference % test_1,pre_spends TTest p-value test_1,...,post_spends TTest pass test_1,pre_spends KSTest p-value test_1,pre_spends KSTest pass test_1,post_spends KSTest p-value test_1,post_spends KSTest pass test_1,mean TTest p-value,mean TTest pass,mean KSTest p-value,mean KSTest pass,mean test score
0,AASplitter┴rs 0┴,487.805672,487.018306,-0.787365,-0.161410,452.102782,452.506439,0.403657,0.089284,0.045150,...,False,0.148992,False,0.964735,False,0.335546,0.5,0.556863,0.0,0.483091
1,AASplitter┴rs 1┴,487.502131,487.331902,-0.170228,-0.034918,451.858488,452.738270,0.879782,0.194703,0.664886,...,False,0.319556,False,0.215823,False,0.476426,0.0,0.267689,0.0,0.337268
2,AASplitter┴rs 2┴,487.553748,487.275719,-0.278029,-0.057025,452.464498,452.136842,-0.327655,-0.072416,0.479308,...,False,0.980091,False,0.778336,False,0.585827,0.0,0.879214,0.0,0.781418
3,AASplitter┴rs 3┴,487.544651,487.289654,-0.254997,-0.052302,451.455436,453.137290,1.681853,0.372540,0.516578,...,True,0.964240,False,0.036004,True,0.279395,0.5,0.500122,0.5,0.426547
4,AASplitter┴rs 4┴,487.493518,487.339850,-0.153668,-0.031522,452.553718,452.054095,-0.499623,-0.110401,0.695790,...,False,0.793024,False,0.800993,False,0.621033,0.0,0.797008,0.0,0.738350
5,AASplitter┴rs 5┴,487.316674,487.516745,0.200071,0.041056,452.441398,452.162139,-0.279258,-0.061723,0.610675,...,False,0.482767,False,0.918082,False,0.673301,0.0,0.700425,0.0,0.691383
6,AASplitter┴rs 6┴,487.307029,487.523820,0.216791,0.044487,452.354799,452.250845,-0.103954,-0.022981,0.581283,...,False,0.831604,False,0.976042,False,0.740691,0.0,0.903823,0.0,0.849446
7,AASplitter┴rs 7┴,487.688681,487.144205,-0.544476,-0.111644,452.704099,451.901396,-0.802703,-0.177313,0.165946,...,False,0.072031,False,0.160198,False,0.249139,0.0,0.116115,0.0,0.160456
8,AASplitter┴rs 8┴,487.478759,487.353829,-0.124930,-0.025628,452.378855,452.226168,-0.152687,-0.033752,0.750601,...,False,0.769603,False,0.575881,False,0.802151,0.0,0.672742,0.0,0.715878
9,AASplitter┴rs 9┴,487.637147,487.198677,-0.438470,-0.089917,452.318281,452.286831,-0.031451,-0.006953,0.264683,...,False,0.423586,False,0.963889,False,0.617192,0.0,0.693737,0.0,0.668222


# AATest with unequal group sizes

AATest can be performed to get a split with unequal the groups of different sizes by using `unequal_size` argument. Also Whelch correction can be applied by adding `t_test_equal_vat=False` argument while initiating AATest instance.

In [39]:
data = Dataset(
    roles={
        "user_id": InfoRole(int),
        "pre_spends": TargetRole(),
        "post_spends": TargetRole(),
        "gender": StratificationRole(str),
    },
    data=create_test_data(),
    session=sp_s,
    backend=BackendsEnum.spark
)
data

,user_id,signup_month,treat,pre_spends,post_spends,age,gender,industry
0,0.0,3.0,1.0,498.0,529.888889,24.0,M,E-commerce
1,1.0,0.0,0.0,511.0,421.555556,67.0,F,E-commerce
2,2.0,7.0,1.0,488.5,465.111111,57.0,F,E-commerce
3,3.0,0.0,0.0,469.5,414.111111,60.0,F,Logistics
4,4.0,0.0,0.0,498.5,417.0,57.0,F,E-commerce
...,...,...,...,...,...,...,...,...
9995,9995.0,9.0,1.0,474.5,454.666667,45.0,M,Logistics
9996,9996.0,0.0,0.0,491.5,417.888889,42.0,M,E-commerce
9997,9997.0,0.0,0.0,481.5,415.555556,53.0,M,Logistics
9998,9998.0,5.0,1.0,489.0,506.111111,21.0,F,Logistics


In [40]:
test = AATest(n_iterations=10, control_size=0.3, t_test_equal_var=False)
result = test.execute(data)

100%|██████████| 10/10 [00:18<00:00,  1.80s/it]
[DEBUG _compute_weighted_pvalues] pval_cols = ['pre_spends TTest p-value test_1', 'post_spends TTest p-value test_1', 'pre_spends KSTest p-value test_1', 'post_spends KSTest p-value test_1', 'mean TTest p-value all', 'mean KSTest p-value all']
[DEBUG _compute_weighted_pvalues] col='pre_spends TTest p-value test_1', lookup_key='pre_spends TTest test_1', weight=0.95, NaN=0/10, values=[0.615403820256988, 0.5365187352940073, 0.6809130508812717, 0.4901968089364088, 0.9002734340038776, 0.2981553224494135, 0.17237984182049168, 0.3852983526113555, 0.08239422151407477, 0.9237188814334233]
[DEBUG _compute_weighted_pvalues] col='post_spends TTest p-value test_1', lookup_key='post_spends TTest test_1', weight=0.95, NaN=0/10, values=[0.5223823822000859, 0.6876465276660896, 0.054889555881304085, 0.8226585374402341, 0.24852544674070165, 0.6073087687559546, 0.41162794181438667, 0.3095857843925785, 0.2060008867701033, 0.527027117967302]
[DEBUG _compute_we

In [41]:
result.best_split.data.groupby("split").agg("count")

,user_id,signup_month,treat,pre_spends,post_spends,age,gender,industry
split,,,,,,,,
test_1,6347,6347,6347,6347,6347,6347,6347,6347
control,2654,2654,2654,2654,2654,2654,2654,2654


In [42]:
result.best_split_statistic

,feature,group,control mean,test mean,difference,difference %,TTest pass,TTest p-value,KSTest pass,KSTest p-value
0,post_spends,test_1,452.115102,451.911538,-0.203564,-0.045025,OK,0.822659,OK,0.936276
1,pre_spends,test_1,487.530487,487.230972,-0.299515,-0.061435,OK,0.490197,OK,0.816359


# AAnTest

AAnTest is an extension of AATest that allows to split the dataset into several test groups, additionally to the control group.

In [43]:
data = Dataset(
    roles={
        "user_id": InfoRole(int),
        "pre_spends": TargetRole(),
        "post_spends": TargetRole(),
        "gender": StratificationRole(str),
    },
    data=create_test_data(),
    session=sp_s,
    backend=BackendsEnum.spark
)
data

,user_id,signup_month,treat,pre_spends,post_spends,age,gender,industry
0,0.0,5.0,1.0,484.0,502.333333,65.0,M,Logistics
1,1.0,0.0,0.0,488.5,416.777778,30.0,M,Logistics
2,2.0,4.0,1.0,489.5,499.888889,62.0,M,E-commerce
3,3.0,0.0,0.0,462.5,419.555556,43.0,M,E-commerce
4,4.0,3.0,1.0,481.0,517.333333,51.0,M,Logistics
...,...,...,...,...,...,...,...,...
9995,9995.0,0.0,0.0,493.0,421.888889,66.0,F,E-commerce
9996,9996.0,11.0,1.0,475.5,427.111111,41.0,F,E-commerce
9997,9997.0,0.0,0.0,488.5,426.0,64.0,M,E-commerce
9998,9998.0,0.0,0.0,516.5,419.888889,33.0,M,E-commerce


In [44]:
test = AATest(groups_sizes=[0.3, 0.2, 0.2, 0.3])
result = test.execute(data)

100%|██████████| 10/10 [00:18<00:00,  1.85s/it]
[DEBUG _compute_weighted_pvalues] pval_cols = ['pre_spends TTest p-value test_1', 'post_spends TTest p-value test_1', 'pre_spends TTest p-value test_2', 'post_spends TTest p-value test_2', 'pre_spends TTest p-value test_3', 'post_spends TTest p-value test_3', 'pre_spends KSTest p-value test_1', 'pre_spends KSTest p-value test_2', 'pre_spends KSTest p-value test_3', 'post_spends KSTest p-value test_1', 'post_spends KSTest p-value test_2', 'post_spends KSTest p-value test_3', 'mean TTest p-value all', 'mean KSTest p-value all']
[DEBUG _compute_weighted_pvalues] col='pre_spends TTest p-value test_1', lookup_key='pre_spends TTest test_1', weight=0.95, NaN=0/10, values=[0.9019733376524891, 0.4366006496044048, 0.9785447095401334, 0.4472006114952731, 0.8845014035360621, 0.6047318845694676, 0.7264329655079214, 0.2986965623610174, 0.560351703614468, 0.28980534134794045]
[DEBUG _compute_weighted_pvalues] col='post_spends TTest p-value test_1', look

In [45]:
result.best_split.data.groupby("split").agg("count")

,user_id,signup_month,treat,pre_spends,post_spends,age,gender,industry
split,,,,,,,,
control,2731,2731,2731,2731,2731,2731,2731,2731
test_3,2668,2668,2668,2668,2668,2668,2668,2668
test_1,1787,1787,1787,1787,1787,1787,1787,1787
test_2,1815,1815,1815,1815,1815,1815,1815,1815


In [46]:
result.best_split_statistic

,feature,group,control mean,test mean,difference,difference %,TTest pass,TTest p-value,KSTest pass,KSTest p-value
0,post_spends,test_1,451.694273,451.734799,0.040526,0.008972,OK,0.973191,OK,0.875904
1,post_spends,test_2,451.694273,451.858497,0.164224,0.036357,OK,0.891054,OK,0.835454
2,post_spends,test_3,451.694273,452.255464,0.561190,0.124241,OK,0.669822,OK,0.994405
3,pre_spends,test_1,487.353945,487.984445,0.630500,0.129372,OK,0.289805,OK,0.919944
4,pre_spends,test_2,487.353945,487.301721,-0.052224,-0.010716,OK,0.928343,OK,0.979560
5,pre_spends,test_3,487.353945,487.015978,-0.337967,-0.069347,OK,0.597801,OK,0.255459


# AATest with partially pre-defined groups

Certain users can be pre-assigned to either the test or the control group, so that they are not randomly assigned. This can be done using the `ConstGroupRole` role. In order to pre-assign users to the control group they should have a value of `control`, and in the test group they should have a value of `test` in the column with the role `ConstGroupRole`. Users that are not pre-assigned to either the control or the test group should have `None`, so that they will be assigned randomly.

In [47]:
pd_data= create_test_data()
pd_data.loc[pd_data["treat"]==0, "const_grp"] = "control"
pd_data.loc[pd_data["treat"]==1, "const_grp"] = "test"
pd_data.loc[2000:, "const_grp"] = None

data = Dataset(
    roles={
        "user_id": InfoRole(int),
        "const_grp": ConstGroupRole(str),
        "pre_spends": TargetRole(),
        "post_spends": TargetRole(),
        "gender": StratificationRole(str),
        "industry": TargetRole(str),
    }, data=pd_data,
    session=sp_s,
    backend=BackendsEnum.spark
)
data

,user_id,signup_month,treat,pre_spends,post_spends,age,gender,industry,const_grp
0,0.0,1.0,1.0,546.5,512.111111,34.0,F,Logistics,test
1,1.0,4.0,1.0,488.5,517.777778,38.0,F,E-commerce,test
2,2.0,0.0,0.0,460.0,404.888889,29.0,M,Logistics,control
3,3.0,0.0,0.0,486.0,424.444444,29.0,M,Logistics,control
4,4.0,0.0,0.0,517.5,425.777778,36.0,F,E-commerce,control
...,...,...,...,...,...,...,...,...,...
9995,9995.0,6.0,1.0,478.0,480.444444,34.0,F,E-commerce,None
9996,9996.0,7.0,1.0,483.0,481.555556,63.0,M,E-commerce,None
9997,9997.0,0.0,0.0,492.5,419.555556,29.0,F,E-commerce,None
9998,9998.0,2.0,1.0,498.0,513.666667,39.0,M,E-commerce,None


In [48]:
test = AATest(n_iterations=1)
result = test.execute(data)

100%|██████████| 1/1 [00:02<00:00,  2.08s/it]
[DEBUG _compute_weighted_pvalues] pval_cols = ['pre_spends TTest p-value control', 'post_spends TTest p-value control', 'pre_spends TTest p-value test_1', 'post_spends TTest p-value test_1', 'pre_spends KSTest p-value control', 'pre_spends KSTest p-value test_1', 'post_spends KSTest p-value control', 'post_spends KSTest p-value test_1', 'industry Chi2Test p-value control', 'industry Chi2Test p-value test_1', 'mean TTest p-value all', 'mean Chi2Test p-value all', 'mean KSTest p-value all']
[DEBUG _compute_weighted_pvalues] col='pre_spends TTest p-value control', lookup_key='pre_spends TTest control', weight=0.95, NaN=0/1, values=[0.4485666943252583]
[DEBUG _compute_weighted_pvalues] col='post_spends TTest p-value control', lookup_key='post_spends TTest control', weight=0.95, NaN=0/1, values=[0.7491305496468442]
[DEBUG _compute_weighted_pvalues] col='pre_spends TTest p-value test_1', lookup_key='pre_spends TTest test_1', weight=0.95, NaN=0/1,

In [49]:
result.resume

,feature,group,TTest aa test,KSTest aa test,Chi2Test aa test,TTest best split,KSTest best split,Chi2Test best split,result,control mean,test mean,difference,difference %
0,industry,control,None,None,OK,None,None,OK,OK,NaN,NaN,NaN,NaN
1,industry,test_1,None,None,OK,None,None,OK,OK,NaN,NaN,NaN,NaN
2,post_spends,control,OK,OK,None,OK,OK,None,OK,451.778357,452.076500,0.298143,0.065993
3,post_spends,test_1,OK,OK,None,OK,OK,None,OK,451.778357,451.297366,-0.480992,-0.106466
4,pre_spends,control,OK,OK,None,OK,OK,None,OK,487.375652,487.035985,-0.339667,-0.069693
5,pre_spends,test_1,OK,OK,None,OK,OK,None,OK,487.375652,487.150472,-0.225180,-0.046203


In [50]:
result.best_split

,user_id,signup_month,treat,pre_spends,post_spends,age,gender,industry,const_grp,split
2,2,0.0,0.0,460.0,404.888889,29.0,M,Logistics,control,None
4,4,0.0,0.0,517.5,425.777778,36.0,F,E-commerce,control,None
5,5,0.0,0.0,480.5,403.888889,23.0,F,E-commerce,control,None
8,8,8.0,1.0,485.5,453.666667,37.0,F,E-commerce,test,None
12,12,0.0,0.0,452.5,408.555556,64.0,F,E-commerce,control,None
...,...,...,...,...,...,...,...,...,...,...
8996,9146,0.0,0.0,513.5,433.888889,41.0,M,E-commerce,None,control
8997,9156,0.0,0.0,498.5,428.555556,41.0,F,Logistics,None,control
8998,9158,0.0,0.0,480.0,421.0,40.0,F,Logistics,None,test_1
8999,9159,0.0,0.0,465.5,422.111111,48.0,F,Logistics,None,test_1


## Common issues and tips

- **Missing roles**: Make sure all target variables are assigned `TargetRole`. Columns without roles may cause silent failure.
- **Stratification**: If your dataset contains categorical features (e.g. `gender`, `region`) that may affect the outcome, use `StratificationRole` and enable `stratification=True` in `AATest(...)`.
- **Imbalanced categories**: If some categories have too few samples, stratified splits may become unstable. Consider filtering or merging rare categories.
- **Random fluctuations**: On small datasets, it's normal to see occasional `NOT OK` results. Use more iterations (e.g. `n_iterations=50`) for stability.
- **Missing values**: NaNs in stratification columns may be treated as separate categories. Clean or fill missing values before stratified AA tests.